# 🔍 Semantic Search Engine — Demo Notebook

This notebook walks through the full pipeline:
1. Load & chunk documents
2. Generate embeddings (Sentence Transformers)
3. Store in FAISS
4. Semantic search vs keyword search comparison
5. Similarity score visualization

In [ ]:
# Install dependencies (run once)
# !pip install sentence-transformers faiss-cpu langchain langchain-text-splitters pymupdf

In [ ]:
import sys
sys.path.append('../backend')

from search_engine import SemanticSearchEngine
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print('✅ Imports successful')

## Step 1 — Initialize the engine

In [ ]:
engine = SemanticSearchEngine(
    model_name='all-MiniLM-L6-v2',
    chunk_size=300,
    chunk_overlap=30
)
print(f'Embedding dimension: {engine.embedding_dim}')

## Step 2 — Add sample documents

In [ ]:
sample_docs = [
    (
        'ai_overview.txt',
        """Artificial Intelligence (AI) is the simulation of human intelligence processes by machines.
Machine learning is a subset of AI where systems learn from data automatically.
Deep learning uses neural networks with many layers to process complex patterns.
Natural language processing (NLP) enables computers to understand human language.
Computer vision allows machines to interpret and understand visual information from the world.
Reinforcement learning trains agents to make decisions by rewarding correct actions.
Transfer learning reuses models trained on one task for a different but related task."""
    ),
    (
        'python_basics.txt',
        """Python is a high-level, interpreted programming language known for its simplicity.
Python uses indentation to define code blocks instead of curly braces.
Lists, tuples, sets and dictionaries are the main built-in data structures.
Functions are defined using the def keyword and can return multiple values.
Python supports object-oriented, functional and procedural programming styles.
The pip package manager is used to install third-party libraries easily.
Virtual environments isolate project dependencies from the global Python installation."""
    ),
    (
        'climate_change.txt',
        """Climate change refers to long-term shifts in global temperatures and weather patterns.
Human activities, primarily burning fossil fuels, release greenhouse gases into the atmosphere.
Carbon dioxide and methane trap heat from the sun causing the greenhouse effect.
Rising sea levels threaten coastal cities and low-lying island nations.
Renewable energy sources like solar and wind power can reduce carbon emissions.
Deforestation contributes to climate change by removing carbon-absorbing trees.
International agreements like the Paris Accord aim to limit global warming to 1.5°C."""
    ),
]

for filename, text in sample_docs:
    chunks = engine.add_document(text.encode(), filename=filename, content_type='text/plain')
    print(f'✅ {filename}: {chunks} chunks')

print(f'\nTotal vectors in FAISS index: {engine.get_doc_count()}')

## Step 3 — Semantic search

In [ ]:
query = 'how do machines learn from data'

results = engine.semantic_search(query, top_k=5)

print(f'Query: "{query}"\n')
print('=== Semantic Search Results ===')
for r in results:
    print(f"\n[Rank {r['rank']}] Score: {r['score']:.4f} | Source: {r['source']}")
    print(f"  {r['text'][:150]}...")

## Step 4 — Keyword search (for comparison)

In [ ]:
keyword_results = engine.keyword_search(query, top_k=5)

print(f'Query: "{query}"\n')
print('=== Keyword Search Results ===')
for r in keyword_results:
    print(f"\n[Rank {r['rank']}] Score: {r['score']:.4f} | Source: {r['source']}")
    print(f"  {r['text'][:150]}...")

## Step 5 — Visualize similarity scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Search Results — Query: "{query}"', fontsize=13, fontweight='bold')

def plot_results(ax, results, title, color):
    if not results:
        ax.text(0.5, 0.5, 'No results', ha='center', va='center')
        return
    labels = [f"[{r['rank']}] {r['source']}\n{r['text'][:40]}..." for r in results]
    scores = [r['score'] for r in results]
    y = range(len(results))
    bars = ax.barh(y, scores, color=color, alpha=0.8, edgecolor='white')
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Similarity Score')
    ax.set_title(title, fontweight='bold')
    ax.invert_yaxis()
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plot_results(axes[0], results, '🧠 Semantic Search', '#6366f1')
plot_results(axes[1], keyword_results, '🔑 Keyword Search', '#f59e0b')

plt.tight_layout()
plt.savefig('search_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as search_comparison.png')

## Step 6 — Try different queries and observe the difference

In [ ]:
test_queries = [
    'global warming and temperature rise',       # maps to climate_change.txt
    'installing packages in python',             # maps to python_basics.txt
    'neural networks and pattern recognition',   # maps to ai_overview.txt
    'clean energy sources to reduce pollution',  # synonym-heavy — semantic wins
]

for q in test_queries:
    sem = engine.semantic_search(q, top_k=1)
    kw  = engine.keyword_search(q, top_k=1)
    sem_src = sem[0]['source'] if sem else 'none'
    kw_src  = kw[0]['source']  if kw  else 'none'
    match = '✅' if sem_src == kw_src else '⚡ DIFFER'
    print(f"{match} Query: '{q}'")
    print(f"   Semantic → {sem_src} ({sem[0]['score']:.3f})")
    print(f"   Keyword  → {kw_src}  ({kw[0]['score']:.3f})\n")